# 112. Voronoi分割の汎化性能評価（ドメイン外データ）

## 目的
- NB111でWikipedia 10Kデータ上で学習したVoronoiセントロイド（k-meansの超平面）を、
  **ドメインの異なる非公開データ**に適用した場合の汎化性能を検証
- Wikipedia（汎用ドメイン）で学習した分割が、特定ドメインのデータでも有効かを確認

## 実験設計
1. Wikipedia 10K（E5-base, 768D）でk-meansセントロイドを学習（NB111と同条件）
2. 非公開データ（ドメイン特化, 2,789件）にE5-base embeddingを生成
3. Wikipedia学習済みセントロイドを非公開データに適用してVoronoi検索を評価
4. 非公開データ自身で学習したセントロイドとの比較（upper bound）

## NB111の参考値（Wikipedia 10K, E5-base）
| 構成 | EN R@10 | JA R@10 | 削減率 |
|------|---------|---------|--------|
| Voronoi(C=128, P=5) | 85.9% | 92.3% | 95.3%/94.7% |
| Voronoi(C=64, P=5) | 90.3% | 94.6% | 91.1%/91.4% |
| Voronoi(C=32, P=5) | 93.2% | 96.1% | 82.3%/82.8% |

## 0. セットアップ

In [1]:
import sys
import numpy as np
import time
import duckdb
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../src')
from itq_lsh import ITQLSH

DATA_DIR = Path('../data')
np.random.seed(42)

N_QUERIES = 100
TOP_K = 10
print(f'Configuration: N_QUERIES={N_QUERIES}, TOP_K={TOP_K}')

Configuration: N_QUERIES=100, TOP_K=10


## 1. Wikipedia学習データのロードとVoronoiセントロイド学習

NB111と同じ条件で、Wikipedia 10K E5-base embeddingからk-meansセントロイドを学習する。

In [2]:
# Wikipedia E5-base embeddings（NB111と同じデータ）
wiki_emb_en = np.load(DATA_DIR / '10k_e5_base_en_embeddings.npy')
wiki_emb_ja = np.load(DATA_DIR / '10k_e5_base_ja_embeddings.npy')
print(f'Wikipedia EN: {wiki_emb_en.shape}')
print(f'Wikipedia JA: {wiki_emb_ja.shape}')

# EN+JA混合でもセントロイドを学習（非公開データはJA/EN混在のため）
wiki_emb_mixed = np.vstack([wiki_emb_en, wiki_emb_ja])
print(f'Wikipedia Mixed: {wiki_emb_mixed.shape}')


def build_voronoi(embeddings, n_clusters, random_state=42):
    """k-meansでVoronoi分割を構築する（NB111と同じ）"""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    emb_normed = embeddings / norms
    
    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        batch_size=2048,
        n_init=3
    )
    labels = kmeans.fit_predict(emb_normed)
    
    centroids = kmeans.cluster_centers_
    centroids_normed = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    
    partition_indices = {}
    for c in range(n_clusters):
        partition_indices[c] = np.where(labels == c)[0]
    
    sizes = [len(v) for v in partition_indices.values()]
    print(f'  n_clusters={n_clusters}: '
          f'size: mean={np.mean(sizes):.1f}, '
          f'min={np.min(sizes)}, max={np.max(sizes)}, '
          f'std={np.std(sizes):.1f}')
    
    return centroids_normed, labels, partition_indices


# Wikipediaデータでセントロイド学習
cluster_sizes = [32, 64, 128, 256]
wiki_centroids = {}

for source_name, emb in [('EN', wiki_emb_en), ('JA', wiki_emb_ja), ('Mixed', wiki_emb_mixed)]:
    print(f'\nWikipedia {source_name}:')
    wiki_centroids[source_name] = {}
    for n_c in cluster_sizes:
        centroids, labels, partitions = build_voronoi(emb, n_c)
        wiki_centroids[source_name][n_c] = {
            'centroids': centroids,
            'labels': labels,
            'partitions': partitions,
        }

Wikipedia EN: (10000, 768)
Wikipedia JA: (9990, 768)
Wikipedia Mixed: (19990, 768)

Wikipedia EN:


  n_clusters=32: size: mean=312.5, min=105, max=526, std=119.5


  n_clusters=64: size: mean=156.2, min=43, max=266, std=59.6


  n_clusters=128: size: mean=78.1, min=2, max=259, std=43.4


  n_clusters=256: size: mean=39.1, min=1, max=163, std=33.5

Wikipedia JA:


...

  n_clusters=256: size: mean=39.0, min=1, max=244, std=37.4

Wikipedia Mixed:


  n_clusters=32: size: mean=624.7, min=243, max=1186, std=250.8


  n_clusters=64: size: mean=312.3, min=1, max=780, std=142.3


  n_clusters=128: size: mean=156.2, min=1, max=485, std=83.8


  n_clusters=256: size: mean=78.1, min=1, max=303, std=64.0


## 2. 非公開データのロードとE5-base Embedding生成

In [3]:
# 非公開データの読み込み
con = duckdb.connect(str(DATA_DIR / "resou_data.duckdb"), read_only=True)
private_df = con.execute("""
    SELECT id, combined_text, language
    FROM documents
    WHERE combined_text IS NOT NULL AND LENGTH(combined_text) > 0
    ORDER BY id
""").fetchdf()
con.close()

print(f'非公開データ: {len(private_df)} 件')
print(f'  JA: {(private_df["language"] == "ja").sum()} 件')
print(f'  EN: {(private_df["language"] == "en").sum()} 件')
print(f'  テキスト長: mean={private_df["combined_text"].str.len().mean():.0f}, '
      f'max={private_df["combined_text"].str.len().max()}')

非公開データ: 2789 件
  JA: 1614 件
  EN: 1175 件
  テキスト長: mean=517, max=2752


In [4]:
# E5-base Embeddingの生成
from sentence_transformers import SentenceTransformer
import gc, torch

E5_MODEL_ID = "intfloat/multilingual-e5-base"
SAVE_PATH = DATA_DIR / "112_private_e5_base_embeddings.npy"

if SAVE_PATH.exists():
    print(f"キャッシュ済み: {SAVE_PATH}")
    emb_private_all = np.load(SAVE_PATH)
else:
    print(f"Loading {E5_MODEL_ID}...")
    model = SentenceTransformer(E5_MODEL_ID, device="cuda")
    
    # E5-base は "query: " / "passage: " プレフィックスを使用
    texts = ["passage: " + t for t in private_df['combined_text'].tolist()]
    print(f"Encoding {len(texts)} documents...")
    emb_private_all = model.encode(
        texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True
    )
    
    np.save(SAVE_PATH, emb_private_all)
    print(f"Saved: {SAVE_PATH}")
    
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f'Embedding shape: {emb_private_all.shape}')

# 言語別に分割
ja_mask = (private_df['language'] == 'ja').values
en_mask = (private_df['language'] == 'en').values
emb_private_ja = emb_private_all[ja_mask]
emb_private_en = emb_private_all[en_mask]
print(f'  JA: {emb_private_ja.shape}')
print(f'  EN: {emb_private_en.shape}')

Loading intfloat/multilingual-e5-base...


Encoding 2789 documents...


Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Saved: ../data/112_private_e5_base_embeddings.npy
Embedding shape: (2789, 768)
  JA: (1614, 768)
  EN: (1175, 768)


## 3. 評価関数

In [5]:
def get_ground_truth(embeddings, qi, top_k=10):
    """ブルートフォースでcosine類似度のtop-kを取得"""
    cos_sims = cosine_similarity(embeddings[qi:qi+1], embeddings)[0]
    cos_sims[qi] = -1
    return set(np.argsort(cos_sims)[-top_k:])


def apply_voronoi_and_evaluate(embeddings, centroids, n_probes,
                               n_queries=100, top_k=10, seed=42):
    """外部セントロイドを適用してVoronoi検索を評価する。
    
    セントロイドは別データで学習済み（汎化テスト用）。
    """
    rng = np.random.default_rng(seed)
    n_queries = min(n_queries, len(embeddings) // 2)
    query_indices = rng.choice(len(embeddings), n_queries, replace=False)
    
    # 正規化
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    emb_normed = embeddings / norms
    
    # 全ベクトルをセントロイドに割り当て
    labels = np.argmax(emb_normed @ centroids.T, axis=1)
    partitions = {}
    for c in range(len(centroids)):
        partitions[c] = np.where(labels == c)[0]
    
    # パーティションサイズ統計
    sizes = [len(v) for v in partitions.values() if len(v) > 0]
    n_empty = sum(1 for v in partitions.values() if len(v) == 0)
    
    filter_recalls = []
    final_recalls = []
    candidate_counts = []
    times = []
    
    for qi in query_indices:
        gt = get_ground_truth(embeddings, qi, top_k)
        q_emb = emb_normed[qi]
        
        start = time.time()
        
        sims_to_centroids = centroids @ q_emb
        top_centroids = np.argsort(-sims_to_centroids)[:n_probes]
        
        candidates = []
        for c in top_centroids:
            if c in partitions and len(partitions[c]) > 0:
                candidates.append(partitions[c])
        
        if candidates:
            candidates = np.concatenate(candidates)
            candidates = candidates[candidates != qi]
        else:
            candidates = np.array([], dtype=int)
        
        candidate_counts.append(len(candidates))
        filter_recalls.append(len(gt & set(candidates)) / top_k)
        
        if len(candidates) > 0:
            cand_sims = cosine_similarity(embeddings[qi:qi+1], embeddings[candidates])[0]
            top_in_cand = candidates[np.argsort(-cand_sims)[:top_k]]
            final_recalls.append(len(gt & set(top_in_cand)) / top_k)
        else:
            final_recalls.append(0.0)
        
        times.append(time.time() - start)
    
    return {
        'candidates': np.mean(candidate_counts),
        'candidates_std': np.std(candidate_counts),
        'reduction': 1 - np.mean(candidate_counts) / len(embeddings),
        'filter_recall': np.mean(filter_recalls),
        'recall_at_k': np.mean(final_recalls),
        'time_ms': np.mean(times) * 1000,
        'n_empty_partitions': n_empty,
        'partition_size_mean': np.mean(sizes) if sizes else 0,
        'partition_size_std': np.std(sizes) if sizes else 0,
    }

## 4. 汎化テスト: Wikipediaセントロイド → 非公開データ

Wikipedia（EN/JA/Mixed）で学習したセントロイドを非公開データ（全体/JA/EN）に適用し、
非公開データ自身で学習したセントロイド（upper bound）と比較する。

In [6]:
probe_sizes = [1, 2, 3, 5, 10]

# 非公開データのサブセット定義
private_datasets = {
    'All(JA+EN)': emb_private_all,
    'JA': emb_private_ja,
    'EN': emb_private_en,
}

# セントロイドソース定義
centroid_sources = {
    'Wiki-EN': 'EN',
    'Wiki-JA': 'JA',
    'Wiki-Mixed': 'Mixed',
}

all_results = {}

for priv_name, priv_emb in private_datasets.items():
    print(f'\n{"="*80}')
    print(f'非公開データ: {priv_name} (N={len(priv_emb)})')
    print(f'{"="*80}')
    
    results = []
    
    # 1. 各Wikipediaセントロイドを適用
    for src_label, src_key in centroid_sources.items():
        for n_c in cluster_sizes:
            centroids = wiki_centroids[src_key][n_c]['centroids']
            for n_p in probe_sizes:
                if n_p > n_c:
                    continue
                r = apply_voronoi_and_evaluate(
                    priv_emb, centroids, n_probes=n_p,
                    n_queries=N_QUERIES, top_k=TOP_K
                )
                r['source'] = src_label
                r['n_clusters'] = n_c
                r['n_probes'] = n_p
                r['name'] = f'{src_label}(C={n_c},P={n_p})'
                results.append(r)
    
    # 2. 非公開データ自身でセントロイド学習（upper bound）
    print(f'\n  Self-trained (upper bound):')
    for n_c in cluster_sizes:
        centroids_self, _, _ = build_voronoi(priv_emb, n_c)
        for n_p in probe_sizes:
            if n_p > n_c:
                continue
            r = apply_voronoi_and_evaluate(
                priv_emb, centroids_self, n_probes=n_p,
                n_queries=N_QUERIES, top_k=TOP_K
            )
            r['source'] = 'Self'
            r['n_clusters'] = n_c
            r['n_probes'] = n_p
            r['name'] = f'Self(C={n_c},P={n_p})'
            results.append(r)
    
    all_results[priv_name] = results
    
    # 主要構成の結果テーブル（C=64, C=128のみ表示）
    print(f'\n--- 主要構成の結果 ---')
    print(f'{"Name":<30} {"Cands":>7} {"Reduc":>8} {"FiltR":>8} {"R@10":>8} {"Empty":>6}')
    print('-' * 72)
    for r in results:
        if r['n_clusters'] in [64, 128] and r['n_probes'] in [3, 5, 10]:
            print(f'{r["name"]:<30} {r["candidates"]:>6.0f} '
                  f'{r["reduction"]*100:>7.1f}% '
                  f'{r["filter_recall"]*100:>7.1f}% '
                  f'{r["recall_at_k"]*100:>7.1f}% '
                  f'{r["n_empty_partitions"]:>5}')


非公開データ: All(JA+EN) (N=2789)



  Self-trained (upper bound):


  n_clusters=32: size: mean=87.2, min=18, max=225, std=51.9


  n_clusters=64: size: mean=43.6, min=5, max=117, std=27.3


  n_clusters=128: size: mean=21.8, min=1, max=97, std=17.1


...

  n_clusters=32: size: mean=36.7, min=7, max=118, std=23.2


  n_clusters=64: size: mean=18.4, min=1, max=65, std=12.3


  n_clusters=128: size: mean=9.2, min=1, max=54, std=9.0


  n_clusters=256: size: mean=4.6, min=1, max=41, std=5.4



--- 主要構成の結果 ---
Name                             Cands    Reduc    FiltR     R@10  Empty
------------------------------------------------------------------------
Wiki-EN(C=64,P=3)                 687    41.6%    90.2%    90.2%    40
Wiki-EN(C=64,P=5)                 908    22.8%    95.7%    95.7%    40
Wiki-EN(C=64,P=10)               1076     8.4%    98.7%    98.7%    40
Wiki-EN(C=128,P=3)                505    57.0%    81.0%    81.0%    97
Wiki-EN(C=128,P=5)                710    39.6%    91.6%    91.6%    97
Wiki-EN(C=128,P=10)               988    15.9%    96.8%    96.8%    97
Wiki-JA(C=64,P=3)                 444    62.2%    69.0%    69.0%    39
Wiki-JA(C=64,P=5)                 649    44.7%    86.5%    86.5%    39
Wiki-JA(C=64,P=10)                994    15.4%    97.6%    97.6%    39
Wiki-JA(C=128,P=3)                517    56.0%    75.2%    75.2%    99
Wiki-JA(C=128,P=5)                681    42.1%    87.0%    87.0%    99
Wiki-JA(C=128,P=10)               890    24.2%    96.0% 

## 5. 汎化率の定量分析

Wikipedia学習セントロイドのR@10を、Self-trainedのR@10で割った「汎化率」を算出。
汎化率が1.0に近いほど、Wikipediaセントロイドがドメイン外でも有効。

In [7]:
print('='*80)
print('汎化率分析: Wikipedia学習 vs Self-trained')
print('='*80)

for priv_name in ['All(JA+EN)', 'JA', 'EN']:
    results = all_results[priv_name]
    
    print(f'\n--- 非公開データ: {priv_name} ---')
    print(f'{"Config":<18} {"Wiki-EN":>10} {"Wiki-JA":>10} {"Wiki-Mix":>10} '
          f'{"Self":>10} {"Best Wiki/Self":>14}')
    print('-' * 76)
    
    for n_c in [32, 64, 128]:
        for n_p in [3, 5, 10]:
            # Self-trainedの値
            self_r = [r for r in results 
                     if r['source'] == 'Self' and r['n_clusters'] == n_c and r['n_probes'] == n_p]
            if not self_r:
                continue
            self_recall = self_r[0]['recall_at_k']
            
            # 各Wikiソースの値
            wiki_recalls = {}
            for src in ['Wiki-EN', 'Wiki-JA', 'Wiki-Mixed']:
                wiki_r = [r for r in results
                         if r['source'] == src and r['n_clusters'] == n_c and r['n_probes'] == n_p]
                if wiki_r:
                    wiki_recalls[src] = wiki_r[0]['recall_at_k']
            
            best_wiki = max(wiki_recalls.values()) if wiki_recalls else 0
            ratio = best_wiki / self_recall if self_recall > 0 else 0
            
            label = f'C={n_c},P={n_p}'
            vals = [f'{wiki_recalls.get(s, 0)*100:>9.1f}%' for s in ['Wiki-EN', 'Wiki-JA', 'Wiki-Mixed']]
            print(f'{label:<18} {vals[0]} {vals[1]} {vals[2]} '
                  f'{self_recall*100:>9.1f}% '
                  f'{ratio:>13.3f}')

汎化率分析: Wikipedia学習 vs Self-trained

--- 非公開データ: All(JA+EN) ---
Config                Wiki-EN    Wiki-JA   Wiki-Mix       Self Best Wiki/Self
----------------------------------------------------------------------------
C=32,P=3                95.6%      89.3%      94.9%      84.8%         1.127
C=32,P=5                98.2%      96.1%      97.3%      92.3%         1.064
C=32,P=10               99.3%      98.9%      97.6%      95.7%         1.038
C=64,P=3                82.2%      70.8%      89.4%      74.1%         1.206
C=64,P=5                91.7%      86.7%      94.7%      85.2%         1.112
C=64,P=10               96.8%      96.9%      96.9%      93.5%         1.036
C=128,P=3               76.3%      77.5%      90.0%      67.0%         1.343
C=128,P=5               85.4%      85.4%      94.4%      80.0%         1.180
C=128,P=10              95.8%      94.4%      97.0%      90.5%         1.072

--- 非公開データ: JA ---
Config                Wiki-EN    Wiki-JA   Wiki-Mix       Self Best W

## 6. パーティション分布の比較

Wikipediaセントロイドで非公開データを割り当てたときの分布の偏りを確認。
空のパーティション数や、サイズの偏りが汎化性能に影響する。

In [8]:
print('='*80)
print('パーティション分布の比較')
print('='*80)

for priv_name, priv_emb in private_datasets.items():
    print(f'\n--- 非公開データ: {priv_name} (N={len(priv_emb)}) ---')
    
    norms = np.linalg.norm(priv_emb, axis=1, keepdims=True)
    emb_normed = priv_emb / norms
    
    for n_c in [64, 128, 256]:
        print(f'\n  n_clusters={n_c}:')
        print(f'  {"Source":<15} {"Empty":>6} {"Mean":>8} {"Std":>8} {"Min":>6} {"Max":>6} {"CV":>8}')
        print(f'  {"-"*58}')
        
        for src_label, src_key in [('Wiki-EN', 'EN'), ('Wiki-JA', 'JA'), 
                                     ('Wiki-Mixed', 'Mixed'), ('Self', None)]:
            if src_label == 'Self':
                centroids, _, _ = build_voronoi(priv_emb, n_c)
            else:
                centroids = wiki_centroids[src_key][n_c]['centroids']
            
            labels = np.argmax(emb_normed @ centroids.T, axis=1)
            sizes = []
            n_empty = 0
            for c in range(n_c):
                s = np.sum(labels == c)
                sizes.append(s)
                if s == 0:
                    n_empty += 1
            
            sizes_arr = np.array(sizes)
            non_empty = sizes_arr[sizes_arr > 0]
            cv = np.std(non_empty) / np.mean(non_empty) if len(non_empty) > 0 else 0
            
            if src_label != 'Self':
                print(f'  {src_label:<15} {n_empty:>5} {np.mean(non_empty):>7.1f} '
                      f'{np.std(non_empty):>7.1f} {np.min(non_empty):>5} '
                      f'{np.max(non_empty):>5} {cv:>7.3f}')
            else:
                print(f'  {"Self":<15} {n_empty:>5} {np.mean(non_empty):>7.1f} '
                      f'{np.std(non_empty):>7.1f} {np.min(non_empty):>5} '
                      f'{np.max(non_empty):>5} {cv:>7.3f}')

パーティション分布の比較

--- 非公開データ: All(JA+EN) (N=2789) ---

  n_clusters=64:
  Source           Empty     Mean      Std    Min    Max       CV
  ----------------------------------------------------------
  Wiki-EN            37   103.3   211.8     1   804   2.051
  Wiki-JA            27    75.4   141.2     1   563   1.873
  Wiki-Mixed         31    84.5   208.3     1   964   2.464


  n_clusters=64: size: mean=43.6, min=5, max=117, std=27.3
  Self                0    43.6    27.2     5   117   0.625

  n_clusters=128:
  Source           Empty     Mean      Std    Min    Max       CV
  ----------------------------------------------------------
  Wiki-EN            90    73.4   183.2     1  1065   2.496
  Wiki-JA            71    48.9   157.1     1  1115   3.211
  Wiki-Mixed         77    54.7   175.3     1   993   3.206


  n_clusters=128: size: mean=21.8, min=1, max=97, std=17.1
  Self                0    21.8    17.1     1    97   0.786

  n_clusters=256:
  Source           Empty     Mean      Std    Min    Max       CV
  ----------------------------------------------------------
  Wiki-EN           208    58.1   152.7     1   753   2.629
  Wiki-JA           174    34.0   119.5     1   765   3.514
  Wiki-Mixed        189    41.6   117.9     1   713   2.833


  n_clusters=256: size: mean=10.9, min=1, max=62, std=11.1
  Self                0    10.9    11.1     1    62   1.016

--- 非公開データ: JA (N=1614) ---

  n_clusters=64:
  Source           Empty     Mean      Std    Min    Max       CV
  ----------------------------------------------------------
  Wiki-EN            43    76.9   121.5     1   466   1.580
  Wiki-JA            29    46.1    94.3     1   389   2.046
  Wiki-Mixed         45    84.9   174.8     1   695   2.057


  n_clusters=64: size: mean=25.2, min=3, max=77, std=17.4
  Self                0    25.2    17.4     3    77   0.690

  n_clusters=128:
  Source           Empty     Mean      Std    Min    Max       CV
  ----------------------------------------------------------
  Wiki-EN            94    47.5   122.0     1   700   2.569
  Wiki-JA            73    29.3   106.0     1   771   3.613
  Wiki-Mixed         97    52.1   143.4     1   688   2.754


  n_clusters=128: size: mean=12.6, min=1, max=48, std=10.6
  Self                0    12.6    10.6     1    48   0.842

  n_clusters=256:
  Source           Empty     Mean      Std    Min    Max       CV
  ----------------------------------------------------------
  Wiki-EN           214    38.4   101.6     1   518   2.644
  Wiki-JA           179    21.0    72.4     1   488   3.454
  Wiki-Mixed        211    35.9   109.9     1   713   3.065


  n_clusters=256: size: mean=6.3, min=1, max=55, std=6.2
  Self                0     6.3     6.1     1    54   0.975

--- 非公開データ: EN (N=1175) ---

  n_clusters=64:
  Source           Empty     Mean      Std    Min    Max       CV
  ----------------------------------------------------------
  Wiki-EN            40    49.0   114.5     1   478   2.339
  Wiki-JA            39    47.0    78.4     1   304   1.668
  Wiki-Mixed         50    83.9   246.5     1   964   2.937


  n_clusters=64: size: mean=18.4, min=1, max=65, std=12.3
  Self                0    18.4    12.3     1    64   0.668

  n_clusters=128:
  Source           Empty     Mean      Std    Min    Max       CV
  ----------------------------------------------------------
  Wiki-EN            97    37.9    77.8     1   365   2.054
  Wiki-JA            99    40.5    86.4     1   344   2.132
  Wiki-Mixed        107    56.0   210.8     1   993   3.768


  n_clusters=128: size: mean=9.2, min=1, max=54, std=9.0
  Self                0     9.2     9.0     1    54   0.982

  n_clusters=256:
  Source           Empty     Mean      Std    Min    Max       CV
  ----------------------------------------------------------
  Wiki-EN           226    39.2    75.3     1   318   1.923
  Wiki-JA           221    33.6   101.4     1   552   3.019
  Wiki-Mixed        233    51.1   129.6     1   583   2.536


  n_clusters=256: size: mean=4.6, min=1, max=41, std=5.4
  Self                0     4.6     5.4     1    41   1.184


## 7. NB111との直接比較

同じn_clusters × n_probes構成で、Wikipedia上のR@10と非公開データ上のR@10を比較。

In [9]:
# NB111の結果（Wikipedia 10K, Self-trainedセントロイド）
nb111_wiki = {
    'EN': {
        (32, 3): 0.844, (32, 5): 0.932, (32, 10): 0.973,
        (64, 3): 0.823, (64, 5): 0.903, (64, 10): 0.965,
        (128, 3): 0.777, (128, 5): 0.859, (128, 10): 0.925,
        (256, 3): 0.753, (256, 5): 0.819, (256, 10): 0.903,
    },
    'JA': {
        (32, 3): 0.907, (32, 5): 0.961, (32, 10): 0.991,
        (64, 3): 0.893, (64, 5): 0.946, (64, 10): 0.977,
        (128, 3): 0.873, (128, 5): 0.923, (128, 10): 0.960,
        (256, 3): 0.829, (256, 5): 0.885, (256, 10): 0.942,
    },
}

print('='*80)
print('NB111 (Wikipedia) vs NB112 (非公開データ) 直接比較')
print('='*80)

# 非公開データのWiki-Mixedセントロイド結果を使用（最も汎用的）
for priv_name in ['All(JA+EN)', 'JA', 'EN']:
    results = all_results[priv_name]
    
    print(f'\n--- 非公開データ: {priv_name} ---')
    print(f'{"Config":<16} {"Wiki(NB111)":>12} {"Private(Wiki-Mix)":>18} {"Private(Self)":>14} {"Wiki→Priv Ratio":>16}')
    print('-' * 80)
    
    for n_c in [32, 64, 128]:
        for n_p in [3, 5, 10]:
            # NB111 Wiki結果（ENとJAの平均、またはAll用）
            if priv_name == 'EN':
                wiki_r10 = nb111_wiki['EN'].get((n_c, n_p), None)
            elif priv_name == 'JA':
                wiki_r10 = nb111_wiki['JA'].get((n_c, n_p), None)
            else:
                # All: EN/JAの平均
                en_val = nb111_wiki['EN'].get((n_c, n_p), 0)
                ja_val = nb111_wiki['JA'].get((n_c, n_p), 0)
                wiki_r10 = (en_val + ja_val) / 2
            
            # 非公開データ Wiki-Mixed結果
            priv_wiki = [r for r in results 
                        if r['source'] == 'Wiki-Mixed' and r['n_clusters'] == n_c and r['n_probes'] == n_p]
            priv_wiki_r10 = priv_wiki[0]['recall_at_k'] if priv_wiki else None
            
            # 非公開データ Self結果
            priv_self = [r for r in results
                        if r['source'] == 'Self' and r['n_clusters'] == n_c and r['n_probes'] == n_p]
            priv_self_r10 = priv_self[0]['recall_at_k'] if priv_self else None
            
            if wiki_r10 and priv_wiki_r10:
                ratio = priv_wiki_r10 / wiki_r10
                label = f'C={n_c},P={n_p}'
                print(f'{label:<16} {wiki_r10*100:>11.1f}% '
                      f'{priv_wiki_r10*100:>17.1f}% '
                      f'{priv_self_r10*100 if priv_self_r10 else 0:>13.1f}% '
                      f'{ratio:>15.3f}')

NB111 (Wikipedia) vs NB112 (非公開データ) 直接比較

--- 非公開データ: All(JA+EN) ---
Config            Wiki(NB111)  Private(Wiki-Mix)  Private(Self)  Wiki→Priv Ratio
--------------------------------------------------------------------------------
C=32,P=3                87.5%              94.9%          84.8%           1.084
C=32,P=5                94.7%              97.3%          92.3%           1.028
C=32,P=10               98.2%              97.6%          95.7%           0.994
C=64,P=3                85.8%              89.4%          74.1%           1.042
C=64,P=5                92.5%              94.7%          85.2%           1.024
C=64,P=10               97.1%              96.9%          93.5%           0.998
C=128,P=3               82.5%              90.0%          67.0%           1.091
C=128,P=5               89.1%              94.4%          80.0%           1.059
C=128,P=10              94.2%              97.0%          90.5%           1.029

--- 非公開データ: JA ---
Config            Wiki(NB111)

## 8. まとめ・評価・考察

In [10]:
print('='*80)
print('実験112 総合まとめ')
print('='*80)

print('''
【実験の問い】
Wikipedia 10Kで学習したVoronoiセントロイド（k-meansの超平面）は、
ドメインの異なる非公開データでも有効に汎化するか？

【実験条件】
- 学習データ: Wikipedia 10K (E5-base, 768D) - NB111と同条件
- テストデータ: ドメイン特化 非公開データ 2,789件 (JA: 1,614 / EN: 1,175)
- セントロイドソース: Wiki-EN / Wiki-JA / Wiki-Mixed(EN+JA) / Self(upper bound)
- 評価: Recall@10, 候補削減率, 汎化率(Wiki/Self)

【上記セクションの結果を踏まえた考察はマークダウンセルに記載】
''')

実験112 総合まとめ

【実験の問い】
Wikipedia 10Kで学習したVoronoiセントロイド（k-meansの超平面）は、
ドメインの異なる非公開データでも有効に汎化するか？

【実験条件】
- 学習データ: Wikipedia 10K (E5-base, 768D) - NB111と同条件
- テストデータ: ドメイン特化 非公開データ 2,789件 (JA: 1,614 / EN: 1,175)
- セントロイドソース: Wiki-EN / Wiki-JA / Wiki-Mixed(EN+JA) / Self(upper bound)
- 評価: Recall@10, 候補削減率, 汎化率(Wiki/Self)

【上記セクションの結果を踏まえた考察はマークダウンセルに記載】



### 評価・考察

#### 1. Wikipediaセントロイドは非公開データで「汎化する」どころか、Self-trainedを大幅に上回る

汎化率（Best Wiki / Self）が全構成で**1.0を超え、最大1.70**に達した。
これはNB111のtrain/test分割テスト（汎化率0.95〜1.0）とは全く異なる傾向。

| 構成 | All(JA+EN) | JA | EN |
|------|-----------|-----|-----|
| C=64, P=5 | 1.112 | 1.167 | 1.300 |
| C=128, P=5 | 1.180 | 1.256 | 1.414 |
| C=128, P=10 | 1.072 | 1.118 | 1.182 |

**原因**: 非公開データ（2,789件）はWikipedia（10K〜20K件）に比べて小規模なため、Self-trainedのセントロイドは少数データに過適合し、パーティション内の候補数が極端に少なくなる（例: C=128, Self → mean=21.8件/パーティション）。一方、Wikipediaの広い分布で学習したセントロイドは空間をより均等にカバーし、結果として多くの候補を拾える。

#### 2. Wiki-Mixedが最も安定して高性能

| データ | 最良ソース | 理由 |
|--------|-----------|------|
| All(JA+EN) | Wiki-Mixed | JA/EN混在に対応 |
| JA | Wiki-Mixed or Wiki-JA | 僅差でWiki-Mixed |
| EN | **Wiki-Mixed が圧倒的** | ENデータが少ない（1,175件）ため、Wiki-Mixedの広いカバレッジが効く |

特にENでは、Wiki-Mixed(C=64,P=10)でR@10=**100.0%**を達成（全件正解）。

#### 3. パーティション分布の偏りが大きいが、Recallは高い

| Source | C=128, Empty | CV | R@10(P=5) |
|--------|-------------|-----|-----------|
| Wiki-EN | 90 | 2.50 | 85.4% |
| Wiki-JA | 71 | 3.21 | 85.4% |
| Wiki-Mixed | 77 | 3.21 | 94.4% |
| Self | 0 | 0.79 | 80.0% |

Wikipediaセントロイドでは**空パーティションが60-80%**を占めるが、非公開データが集中する少数のパーティションに多くの候補が入るため、probing時に効率よく候補を収集できる。CV（変動係数）が大きい＝偏りが大きいが、この偏りはむしろ「密集データをまとめて取得する」方向に作用している。

#### 4. 削減率とRecallのトレードオフ

Wikipediaセントロイドを使うと候補数が増加し、削減率は低下する。

| 構成 | Self 削減率 | Wiki-Mixed 削減率 | Self R@10 | Wiki-Mixed R@10 |
|------|-----------|------------------|-----------|----------------|
| C=64, P=5 (All) | 89.6% | 57.4% | 85.2% | **94.7%** |
| C=128, P=5 (All) | 93.7% | 58.6% | 80.0% | **94.4%** |
| C=128, P=10 (All) | 87.9% | 51.7% | 90.5% | **97.0%** |

Self-trainedは削減率が高い（候補が少ない）が、Recallも低い。Wikipediaセントロイドは候補が多いが、その分Recallが非常に高い。**Firestore運用ではRecallが最重要であるため、Wikipediaセントロイドの方が実用的**。

#### 5. NB111との比較: ドメイン外でもWikipediaと同等以上の性能

Wiki→Private Ratioが全構成で**0.98〜1.25**であり、Wikipediaセントロイドは非公開データに対しても同等以上のRecallを実現する。

| 構成 | Wiki(NB111) | Private(Wiki-Mix) | Ratio |
|------|------------|-------------------|-------|
| C=64, P=5 (All) | 92.5% | 94.7% | 1.024 |
| C=128, P=5 (All) | 89.1% | 94.4% | 1.059 |
| C=128, P=10 (EN) | 92.5% | 99.3% | 1.074 |

非公開データの方がR@10が高いのは、データ規模が小さく（2,789件 vs 10K件）、同じ候補数でカバーできる割合が高いため。

#### 6. 結論

1. **Wikipediaで学習したVoronoiセントロイドはドメイン外の非公開データでも極めて有効**。Self-trainedを大幅に上回り、汎化率1.0〜1.7。
2. **Wiki-Mixed（EN+JA混合）が最も汎用性が高い**。多言語混在データに対して最も安定。
3. **小規模データではSelf-trainedが不利**。セントロイドが少数データに適合しすぎ、パーティションが細かくなりすぎてRecallが低下する。
4. **Firestore運用においては、Wikipediaで一度学習したセントロイドを固定し、新規データにそのまま適用する方式が合理的**。セントロイドの再学習は不要。
5. 推奨構成: **Wiki-Mixed, C=64〜128, P=5〜10** → R@10=94〜99%, 候補数=1,000〜1,500件